In [27]:
from pathlib import Path
import csv
from datetime import date
from decimal import Decimal
from collections import Counter

In [28]:
data_path = Path("../data")
if not data_path.exists():
    data_path = Path("data")

print( data_path.exists())


True


In [29]:
with (data_path / "customer.tbl").open(encoding="ascii", newline="") as f:
    lecteur = csv.reader(f, delimiter="|", quoting=csv.QUOTE_NONE)
    l = next(lecteur)

print("Avant :", len(l), "éléments")
if len(l) == 9 and l[-1] == "":
    l.pop()
print("Après :", len(l), "colonnes")
print(l)


Avant : 9 éléments
Après : 8 colonnes
['1', 'Customer#000000001', 'IVhzIApeRb ot,c,E', '15', '25-989-741-2988', '711.56', 'BUILDING', 'to the even, regular platelets. regular, ironic epitaphs nag e']


In [30]:
customer_columns = [
    "C_CUSTKEY", "C_NAME", "C_ADDRESS", "C_NATIONKEY",
    "C_PHONE", "C_ACCTBAL", "C_MKTSEGMENT", "C_COMMENT"
]

orders_columns = [
    "O_ORDERKEY", "O_CUSTKEY", "O_ORDERSTATUS", "O_TOTALPRICE",
    "O_ORDERDATE", "O_ORDERPRIORITY", "O_CLERK", "O_SHIPPRIORITY", "O_COMMENT"
]

lineitem_columns = [
    "L_ORDERKEY", "L_PARTKEY", "L_SUPPKEY", "L_LINENUMBER",
    "L_QUANTITY", "L_EXTENDEDPRICE", "L_DISCOUNT", "L_TAX",
    "L_RETURNFLAG", "L_LINESTATUS", "L_SHIPDATE", "L_COMMITDATE",
    "L_RECEIPTDATE", "L_SHIPINSTRUCT", "L_SHIPMODE", "L_COMMENT"
]

nation_columns = ["N_NATIONKEY", "N_NAME", "N_REGIONKEY", "N_COMMENT"]


In [31]:
def lire_fichier(nom, colonnes):
    lignes = []
    with (data_path / nom).open(encoding="ascii", newline="") as fichier:
        lecteur = csv.reader(fichier, delimiter="|", quoting=csv.QUOTE_NONE)
        for numero, ligne in enumerate(lecteur, start=1):
            if len(ligne) == len(colonnes) + 1 and ligne[-1] == "":
                ligne.pop()
            if len(ligne) != len(colonnes):
                raise ValueError(f"{nom}, ligne {numero} : nombre de colonnes incorrect")
            lignes.append(dict(zip(colonnes, ligne)))
    return lignes

customers = lire_fichier("customer.tbl", customer_columns)
orders = lire_fichier("orders.tbl", orders_columns)
lineitems = lire_fichier("lineitem.tbl", lineitem_columns)
nations = lire_fichier("nation.tbl", nation_columns)

print("Clients :", len(customers))
print("Commandes :", len(orders))
print("Lignes de commande :", len(lineitems))
print("Pays :", len(nations))


Clients : 1500
Commandes : 15000
Lignes de commande : 60175
Pays : 25


In [32]:
total = 0
for chemin in sorted(data_path.glob("*.tbl")):
    with chemin.open(encoding="ascii") as fichier:
        nombre = sum(1 for ligne in fichier)
    print(chemin.name, ":", nombre)
    total += nombre

print("Total :", total)


customer.tbl : 1500
lineitem.tbl : 60175
nation.tbl : 25
orders.tbl : 15000
part.tbl : 2000
partsupp.tbl : 8000
region.tbl : 5
supplier.tbl : 100
Total : 86805


In [33]:
tables = {"customer": customers, "orders": orders, "lineitem": lineitems, "nation": nations}

for nom, lignes in tables.items():
    vides = 0
    for ligne in lignes:
        for valeur in ligne.values():
            if valeur.strip() == "":
                vides += 1
    print(nom, ":", vides, "valeurs vides")


customer : 0 valeurs vides
orders : 0 valeurs vides
lineitem : 0 valeurs vides
nation : 0 valeurs vides


In [34]:
customer_ids = [client["C_CUSTKEY"] for client in customers]
order_ids = [commande["O_ORDERKEY"] for commande in orders]
lineitem_ids = [(ligne["L_ORDERKEY"], ligne["L_LINENUMBER"]) for ligne in lineitems]
nation_ids = [pays["N_NATIONKEY"] for pays in nations]

print("Clés clients répétées :", len(customer_ids) - len(set(customer_ids)))
print("Clés commandes répétées :", len(order_ids) - len(set(order_ids)))
print("Clés lignes répétées :", len(lineitem_ids) - len(set(lineitem_ids)))
print("Clés pays répétées :", len(nation_ids) - len(set(nation_ids)))

for nom, lignes in tables.items():
    lignes_uniques = set(tuple(ligne.values()) for ligne in lignes)
    print(nom, ":", len(lignes) - len(lignes_uniques), "lignes identiques en trop")


Clés clients répétées : 0
Clés commandes répétées : 0
Clés lignes répétées : 0
Clés pays répétées : 0
customer : 0 lignes identiques en trop
orders : 0 lignes identiques en trop
lineitem : 0 lignes identiques en trop
nation : 0 lignes identiques en trop


In [35]:
client_keys = set(customer_ids)
order_keys = set(order_ids)
nation_keys = set(nation_ids)

commandes_sans_client = 0
for commande in orders:
    if commande["O_CUSTKEY"] not in client_keys:
        commandes_sans_client += 1

lignes_sans_commande = 0
for ligne in lineitems:
    if ligne["L_ORDERKEY"] not in order_keys:
        lignes_sans_commande += 1

clients_sans_pays = 0
for client in customers:
    if client["C_NATIONKEY"] not in nation_keys:
        clients_sans_pays += 1

print("Commandes sans client :", commandes_sans_client)
print("Lignes sans commande :", lignes_sans_commande)
print("Clients sans pays :", clients_sans_pays)


Commandes sans client : 0
Lignes sans commande : 0
Clients sans pays : 0


In [36]:
dates_commandes = [date.fromisoformat(commande["O_ORDERDATE"]) for commande in orders]
soldes = [Decimal(client["C_ACCTBAL"]) for client in customers]
prix_commandes = [Decimal(commande["O_TOTALPRICE"]) for commande in orders]
quantites = [int(ligne["L_QUANTITY"]) for ligne in lineitems]
prix_lignes = [Decimal(ligne["L_EXTENDEDPRICE"]) for ligne in lineitems]
remises = [Decimal(ligne["L_DISCOUNT"]) for ligne in lineitems]
taxes = [Decimal(ligne["L_TAX"]) for ligne in lineitems]

print("Dates des commandes :", min(dates_commandes), "à", max(dates_commandes))
print("Soldes clients :", min(soldes), "à", max(soldes))
print("Clients avec solde négatif :", sum(solde < 0 for solde in soldes))
print("Prix des commandes :", min(prix_commandes), "à", max(prix_commandes))
print("Quantités :", min(quantites), "à", max(quantites))
print("Prix des lignes :", min(prix_lignes), "à", max(prix_lignes))
print("Remises :", min(remises), "à", max(remises))
print("Taxes :", min(taxes), "à", max(taxes))

for colonne in ["L_SHIPDATE", "L_COMMITDATE", "L_RECEIPTDATE"]:
    dates = [date.fromisoformat(ligne[colonne]) for ligne in lineitems]
    print(colonne, ":", min(dates), "à", max(dates))


Dates des commandes : 1992-01-01 à 1998-08-02
Soldes clients : -994.79 à 9987.71
Clients avec solde négatif : 139
Prix des commandes : 874.89 à 466001.28
Quantités : 1 à 50
Prix des lignes : 904.00 à 94949.50
Remises : 0.00 à 0.10
Taxes : 0.00 à 0.08
L_SHIPDATE : 1992-01-04 à 1998-11-29
L_COMMITDATE : 1992-02-02 à 1998-10-28
L_RECEIPTDATE : 1992-01-09 à 1998-12-25


In [37]:
segments = Counter(client["C_MKTSEGMENT"] for client in customers)
transports = Counter(ligne["L_SHIPMODE"] for ligne in lineitems)

print("Segments clients :", dict(sorted(segments.items())))
print("Modes de transport :", dict(sorted(transports.items())))

clients_avec_commande = set(commande["O_CUSTKEY"] for commande in orders)
clients_sans_commande = client_keys - clients_avec_commande
print("Clients sans commande :", len(clients_sans_commande))


Segments clients : {'AUTOMOBILE': 302, 'BUILDING': 337, 'FURNITURE': 279, 'HOUSEHOLD': 294, 'MACHINERY': 288}
Modes de transport : {'AIR': 8491, 'FOB': 8641, 'MAIL': 8669, 'RAIL': 8566, 'REG AIR': 8616, 'SHIP': 8482, 'TRUCK': 8710}
Clients sans commande : 500
